# Cinq lignes de plus, et le cycle apparaît · *Five more lines, and the cycle appears*

Notebook compagnon du chapitre **19. Premier script Python : charger une série FRED et la tracer en dix lignes** — [lire l'article](https://nmlab.io/ressources/premier-script-python-fred).
Companion notebook to chapter **19. Your First Python Script: Load a FRED Series and Plot It in Ten Lines** — [read the article](https://nmlab.io/en/ressources/first-python-script-fred).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_data() -> tuple[Series, Series]:
    """PIB réel (GDPC1) et indicateur de récession NBER (USREC), en direct de FRED.
    Real GDP and the NBER recession flag, live from FRED."""
    gdp = nm.load_fred("GDPC1")
    recessions = nm.load_fred("USREC", start=str(gdp.index[0].year))
    return gdp, recessions

gdp, recessions = load_data()


import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Cinq lignes de plus, et le cycle apparaît",
        sub="Le PIB réel avec les récessions du NBER, ajoutées par une boucle",
        ylab="milliards de dollars de 2017",
        bands="bandes grises =\nrécessions (NBER)",
        note="Une petite boucle sur la série USREC, un axvspan par récession, et votre graphique parle comme celui d'un\n"
             "professionnel. Chaque récession y creuse une marche. Source : BEA et NBER via FRED (GDPC1, USREC)."),
    "en": dict(
        title="Five more lines, and the cycle appears",
        sub="U.S. real GDP with NBER recessions, added by a loop",
        ylab="billions of 2017 dollars",
        bands="grey bands =\nrecessions (NBER)",
        note="A small loop over the USREC series, one axvspan per recession, and your chart speaks like a\n"
             "professional's. Every recession carves a step. Source: BEA and NBER via FRED (GDPC1, USREC)."),
}

def build_figure(gdp: Series, recessions: Series, lang: str) -> Figure:
    """PIB réel ombré des récessions du NBER."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1026)
    ax = nm.axes(fig, left=0.12)
    runs = recessions.ne(recessions.shift()).cumsum()
    for _, run in recessions.groupby(runs):
        if run.iloc[0] == 1:
            ax.axvspan(run.index[0], run.index[-1], color=nm.COLORS["edge"], alpha=0.75, linewidth=0)
    ax.plot(gdp.index, gdp, color=nm.COLORS["blue"], linewidth=2.9)
    ax.set_ylim(0, 26_000)
    ax.set_yticks(range(0, 26_000, 5000))
    ax.yaxis.set_major_formatter(nm.thousands(lang))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.text(0.085, 0.80, text["bands"], transform=ax.transAxes, fontsize=21.5,
            color=nm.COLORS["muted"], linespacing=1.55)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(gdp, recessions, LANG)